# Injini — train & evaluate on Kaggle

Ships the DCASE Task 2 first-shot recipe compressed for an Arm phone: a frozen
AudioSet-distilled MobileNetV3 embedder plus a Mahalanobis distance score.
This notebook produces the evaluation table and exports the model artefacts.

**Settings:** Internet ON. Accelerator GPU (P100 or T4). Add data:
`thetraveller/injini-code` and `zeyadzsm/engine-sounds`. Then Run All.

## 1. Workspace from the attached code dataset

In [ ]:
import os, sys, shutil, json, glob, time
SRC = "/kaggle/input/injini-code"
WORK = "/kaggle/working/injini"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
shutil.copytree(SRC, WORK)
os.chdir(WORK)
sys.path.insert(0, os.path.join(WORK, "src"))
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("workspace:", WORK)
print(os.listdir(WORK))

## 2. Dependencies

In [ ]:
!pip -q install onnx onnxruntime hear21passt 2>/dev/null
import torch, numpy as np
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
# the Kaldi mel matrix ships in the dataset, so features.py needs no torchaudio
assert os.path.exists("models/mel_kaldi_128x513.npy"), "mel matrix missing from dataset"

## 3. DCASE 2025 Task 2 development set (Zenodo 15097779, CC BY-NC-SA 4.0)

In [ ]:
t0 = time.time()
!python src/fetch_dcase.py --out data/dcase2025_dev
n = len(glob.glob("data/dcase2025_dev/*/*/*.wav"))
print(f"{n} wav files in {time.time()-t0:.0f}s")
assert n > 1000, "DCASE dev set did not download"

## 4. Reproduce the DCASE autoencoder baseline

In [ ]:
!python src/baseline_ae.py --root data/dcase2025_dev --epochs 100

## 5. Reference ceiling — PaSST transformer embedding + Mahalanobis
The published state of the art in one reproducible form. The phone pipeline is
measured against this.

In [ ]:
!python src/eval_dcase.py --root data/dcase2025_dev --backend passt --scorer maha     --out models/eval_passt_maha.json || echo "passt backend failed, continuing"

## 6. Frozen EfficientAT mn10_as — FP32, then static INT8

In [ ]:
!python src/embedder.py --name mn10_as --out models/injini_mn10_as_fp32.onnx
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn10_as_fp32.onnx --scorer maha     --out models/eval_mn10_fp32_maha.json
!python export/quantize.py --fp32 models/injini_mn10_as_fp32.onnx     --calib-dir data/dcase2025_dev --n-calib 256
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn10_as_int8.onnx --scorer maha     --out models/eval_mn10_int8_maha.json

## 7. Smaller candidate — mn04_as, FP32 and INT8

In [ ]:
!python src/embedder.py --name mn04_as --out models/injini_mn04_as_fp32.onnx
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn04_as_fp32.onnx --scorer maha     --out models/eval_mn04_fp32_maha.json
!python export/quantize.py --fp32 models/injini_mn04_as_fp32.onnx     --calib-dir data/dcase2025_dev --n-calib 256
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn04_as_int8.onnx --scorer maha     --out models/eval_mn04_int8_maha.json

## 8. kNN scorer cross-check (mn10_as FP32)

In [ ]:
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn10_as_fp32.onnx --scorer knn     --out models/eval_mn10_fp32_knn.json

## 9. Supervised fault-ID head (Kaggle engine-sounds, source-disjoint)

In [ ]:
ES = "/kaggle/input/engine-sounds"
!python src/prepare_engine_sounds.py --root {ES} --out data/engine_sounds_manifest.csv
!python src/faultid.py --manifest data/engine_sounds_manifest.csv     --backend onnx:models/injini_mn10_as_fp32.onnx --epochs 80

## 10. Collect the table and export artefacts

In [ ]:
rows = []
for p in sorted(glob.glob("models/eval_*.json")) + sorted(glob.glob("models/faultid_*.json")):
    d = json.load(open(p))
    rows.append({
        "file": os.path.basename(p),
        "backend": d.get("backend") or d.get("system"),
        "scorer": d.get("scorer"),
        "official_score": d.get("official_score"),
        "mean_auc": d.get("mean_auc"),
        "macro_f1": d.get("macro_f1"),
        "per_machine": d.get("per_machine"),
    })
quant = {}
for p in glob.glob("models/*_quant_report.json"):
    quant[os.path.basename(p)] = json.load(open(p))

summary = {"generated": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()),
           "results": rows, "quant": quant}
json.dump(summary, open("/kaggle/working/injini_metrics.json", "w"), indent=2)

for r in rows:
    print(f"{r['file']:34s} official={r['official_score']}  mean_auc={r['mean_auc']}  macro_f1={r['macro_f1']}")
print()
for k, v in quant.items():
    print(k, "->", {kk: v[kk] for kk in ("fp32_mb", "int8_mb", "size_ratio", "approx_macs")})

for f in glob.glob("models/injini_*.onnx") + glob.glob("models/*_quant_report.json") + glob.glob("models/eval_*.json"):
    shutil.copy(f, "/kaggle/working/")
print("\nartefacts in /kaggle/working/")
print(sorted(os.listdir("/kaggle/working")))